In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Métricas
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             RocCurveDisplay)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Reproducibilidad — fijamos semillas en todos los generadores aleatorios
np.random.seed(42)
tf.random.set_seed(42)

# Rutas
PROCESSED = '../data/processed/'
MODELS = '../models/'

# Cargamos features
df = pd.read_pickle(PROCESSED + 'df_features.pkl')

# Cargamos scaler y columnas — los mismos de la fase 3
scaler = joblib.load(MODELS + 'scaler.pkl')
cols_to_scale = joblib.load(MODELS + 'cols_to_scale.pkl')

# Separamos features y target
X = df.drop(columns=['success'])
y = df['success']

# Split estratificado — mismos parámetros que fase 3 para comparación justa
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# SMOTE solo sobre train
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Escalado — fit ya está hecho, solo transformamos
X_train_bal_scaled = X_train_bal.copy()
X_test_scaled = X_test.copy()
X_train_bal_scaled[cols_to_scale] = scaler.transform(X_train_bal[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# Convertimos a arrays numpy — Keras trabaja con arrays, no dataframes
X_train_np = X_train_bal_scaled.values.astype('float32')
X_test_np = X_test_scaled.values.astype('float32')
y_train_np = y_train_bal.values.astype('float32')
y_test_np = y_test.values.astype('float32')

# Verificación
print("TensorFlow version:", tf.__version__)
print("X_train:", X_train_np.shape)
print("X_test: ", X_test_np.shape)
print("y_train balance:", y_train_np.mean().round(3))
print("y_test balance: ", y_test_np.mean().round(3))

I0000 00:00:1781161783.139572  326456 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781161783.150858  326456 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781161784.335790  326456 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781161790.254300  326456 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

TensorFlow version: 2.21.0
X_train: (3900, 28)
X_test:  (646, 28)
y_train balance: 0.5
y_test balance:  0.755


In [ ]:
def build_model(input_dim):
    """
    Construye y compila el modelo MLP.
    input_dim: número de features de entrada
    """
    model = keras.Sequential([

        # Capa de entrada
        layers.Input(shape=(input_dim,)),

        # Bloque 1
        layers.Dense(128),                        # 128 neuronas, sin activación aún
        layers.BatchNormalization(),               # normalizamos activaciones
        layers.Activation('relu'),                 # ahora aplicamos ReLU
        layers.Dropout(0.3),                       # desactivamos 30% aleatoriamente

        # Bloque 2
        layers.Dense(64),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        # Bloque 3
        layers.Dense(32),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),                       # menos dropout en capas finales

        # Capa de salida — 1 neurona con sigmoid → probabilidad entre 0 y 1
        layers.Dense(1, activation='sigmoid')
    ])

    # Compilamos el modelo
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Construimos el modelo
model = build_model(input_dim=X_train_np.shape[1])

# Resumen de la arquitectura
model.summary()